# Assemble Monthly Model Dataset

**Purpose:** Join all monthly-resolution feature tables, fire labels, and the
static feature grid into a single analysis-ready parquet file. Apply vegetation
cleaning and derive the `water_year` column.

**Pipeline Overview:**

| Step | Description |
|------|-------------|
| 1. Merge monthly features | Inner-join weather agg → wind direction → fire labels → static features |
| 2. Confirm domain | Assert no Water / Urban / Agriculture in veg |
| 3. Remove Riparian | Drop grid cells classified as Riparian vegetation |
| 4. Map veg groups | 13 fine classes → 9 groups |
| 5. Drop sparse groups | Remove Desert, Wetland, Barren |
| 6. Add `water_year` | Oct(Y) – Sep(Y+1) → water year Y+1 |
| 7. Save | `Monthly_03152026/monthly_model_data_by_year.parquet` |

**Inputs** (all produced by earlier pipeline notebooks):

| File | Source notebook |
|------|----------------|
| `Monthly/weather_features.parquet` | `03_01` |
| `Monthly/wind_direction_features.parquet` | `03_01` |
| `static_features_add_openmap.parquet` | `02_06` |
| `Fire_Data/…/calfire_fod_fpa_1994_2020_fire_label_monthly.parquet` | `03_03` |

**Output:** `Clean_Data/Model_Data/Evaluation/Features_w_Label/Monthly_03152026/monthly_model_data_by_year.parquet`

**Features used in the model** (from `04_03`):

| Group | Features |
|-------|----------|
| Weather | `dead_fuel_moisture_1000hr/100hr`, `max/min_air_temperature`, `max/min_relative_humidity`, `precipitation_amount`, `specific_humidity`, `surface_downwelling_shortwave_flux_in_air`, `wind_speed`, `SWE`, `LAI` |
| Wind direction | `wind_direction_category` → one-hot `_N/NE/E/SE/S/SW/W/NW` (mode; not octant counts) |
| Vegetation | `veg_group` → one-hot (6 groups after cleaning) |
| Terrain | `slope_avg`, `slope_max` |
| Infrastructure | `road_density_km_km2`, `minor_line_density`, `trans_line_density` |
| Label / lag | `IS_FIRE`, `prev_fire` |

> **Not used in model:** `population_density`,
> `wind_direction_category_*_freq` (octant counts), `fire_attribute`,
> `min/max` variants of weather variables.

## 0. Configuration

Centralized path configuration — **edit this cell only**.

In [1]:
import os

# ====================== EDIT THESE PATHS ======================
PROJECT_ROOT = r"E:\zcao\CA_Wildfire"

# Inputs
WEATHER_FEATURES_PATH = os.path.join(PROJECT_ROOT, "Clean_Data", "Monthly",
                                      "weather_features.parquet")
WIND_FEATURES_PATH    = os.path.join(PROJECT_ROOT, "Clean_Data", "Monthly",
                                      "wind_direction_features.parquet")
FIRE_LABEL_PATH       = os.path.join(PROJECT_ROOT, "Clean_Data", "Fire_Data",
                                      "Extended_Fire_Data",
                                      "calfire_fod_fpa_1994_2020_fire_label_monthly.parquet")
STATIC_FEATURES_PATH  = os.path.join(PROJECT_ROOT, "Clean_Data",
                                      "static_features_add_openmap.parquet")

# Output
DATA_VERSION = 'Monthly_03152026'
OUTPUT_DIR   = os.path.join(PROJECT_ROOT, "Clean_Data", "Model_Data",
                            "Evaluation", "Features_w_Label", DATA_VERSION)
OUTPUT_FILE  = 'monthly_model_data_by_year.parquet'

# Vegetation mapping (13 fine classes → 9 groups)
VEG_MAPPING = {
    'Native Coastal Sage Scrub' : 'Shrub',
    'Native Conifer Forest'     : 'Conifer Forest',
    'Native Grassland'          : 'Grassland',
    'Native Chapparal'          : 'Chaparral',
    'Native Desert'             : 'Desert',
    'Native Inland Scrub'       : 'Shrub',
    'Native Conifer Alpine'     : 'Conifer Alpine',
    'Non-native forest'         : 'Conifer Forest',
    'Non-native grassland'      : 'Grassland',
    'Barren'                    : 'Barren',
    'Native Wetland'            : 'Wetland',
    'Non-native shrub'          : 'Shrub',
    'Native Oak Woodland'       : 'Oak Woodland',
}
REMOVE_VEG_GROUPS = ['Desert', 'Wetland', 'Barren']

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Output: {os.path.join(OUTPUT_DIR, OUTPUT_FILE)}")
for label, path in [
    ('WEATHER_FEATURES', WEATHER_FEATURES_PATH),
    ('WIND_FEATURES',    WIND_FEATURES_PATH),
    ('FIRE_LABEL',       FIRE_LABEL_PATH),
    ('STATIC_FEATURES',  STATIC_FEATURES_PATH),
]:
    print(f"  [{'OK' if os.path.exists(path) else 'MISSING'}] {label}")

Output: E:\zcao\CA_Wildfire\Clean_Data\Model_Data\Evaluation\Features_w_Label\Monthly_03152026\monthly_model_data_by_year.parquet
  [OK] WEATHER_FEATURES
  [OK] WIND_FEATURES
  [OK] FIRE_LABEL
  [OK] STATIC_FEATURES


## 1. Environment Setup

In [2]:
import sys, gc, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from tqdm import tqdm

gc.collect()
print(f"Python : {sys.version.split('|')[0].strip()}")
print(f"pandas : {pd.__version__}")

Python : 3.9.13 (main, Aug 25 2022, 23:51:50) [MSC v.1916 64 bit (AMD64)]
pandas : 2.2.2


---

## 2. Load All Input Tables

Load each monthly feature table and the static grid.
Note that `fire_label` uses `datetime` for `year_month` and must be converted
to `Period` to align with the other tables.

In [3]:
weather_agg     = pd.read_parquet(WEATHER_FEATURES_PATH)
wind_direction  = pd.read_parquet(WIND_FEATURES_PATH)
fire_label      = pd.read_parquet(FIRE_LABEL_PATH)
static_features = pd.read_parquet(STATIC_FEATURES_PATH)

# Align fire_label year_month to Period (matches weather/wind)
fire_label['year_month'] = fire_label['year_month'].dt.to_period('M')

print(f"weather_agg     : {weather_agg.shape}")
print(f"wind_direction  : {wind_direction.shape}")
print(f"fire_label      : {fire_label.shape}")
print(f"static_features : {static_features.shape}")

weather_agg     : (4181379, 40)
wind_direction  : (4181379, 13)
fire_label      : (4214167, 7)
static_features : (13048, 14)


---

## 3. Merge All Tables

Inner-join in order:
1. `weather_agg` + `wind_direction` → on `(lon, lat, year_month)`
2. + `fire_label` → on `(lon, lat, year_month)`
3. + `static_features` → on `(lon, lat)`

> Inner joins ensure only grid cell × month combinations present in
> **all** feature sources are retained.

In [4]:
mod_data = pd.merge(weather_agg, wind_direction,
                    on=['lon','lat','year_month'], how='inner')
print(f"After weather + wind  : {mod_data.shape}")

mod_data = pd.merge(mod_data, fire_label,
                    on=['lon','lat','year_month'], how='inner')
print(f"After + fire label    : {mod_data.shape}")

mod_data = pd.merge(mod_data, static_features,
                    on=['lon','lat'], how='inner')
print(f"After + static grid   : {mod_data.shape}")

del weather_agg, wind_direction, fire_label, static_features
gc.collect()

print(f"\nColumns ({mod_data.shape[1]}):")
print(list(mod_data.columns))

After weather + wind  : (4181379, 50)
After + fire label    : (4168166, 54)
After + static grid   : (4168166, 66)

Columns (66):
['year_month', 'lat', 'lon', 'SWE', 'SWE_min', 'SWE_max', 'dead_fuel_moisture_1000hr', 'dead_fuel_moisture_1000hr_min', 'dead_fuel_moisture_1000hr_max', 'dead_fuel_moisture_100hr', 'dead_fuel_moisture_100hr_min', 'dead_fuel_moisture_100hr_max', 'max_air_temperature', 'max_air_temperature_min', 'max_air_temperature_max', 'max_relative_humidity', 'max_relative_humidity_min', 'max_relative_humidity_max', 'min_air_temperature', 'min_air_temperature_min', 'min_air_temperature_max', 'min_relative_humidity', 'min_relative_humidity_min', 'min_relative_humidity_max', 'precipitation_amount', 'precipitation_amount_min', 'precipitation_amount_max', 'specific_humidity', 'specific_humidity_min', 'specific_humidity_max', 'surface_downwelling_shortwave_flux_in_air', 'surface_downwelling_shortwave_flux_in_air_min', 'surface_downwelling_shortwave_flux_in_air_max', 'wind_speed'

---

## 4. Confirm Domain: No Water / Urban / Agriculture

The veg + subregion filters applied upstream should have removed these classes.
Assert here as a safety check.

In [5]:
bad_veg = mod_data[mod_data['veg'].str.contains('Water|Urban|Agriculture')]
assert bad_veg.empty, f"Found {len(bad_veg)} rows with Water/Urban/Agriculture veg"
print(f"Domain check OK — no Water/Urban/Agriculture rows.")
print(f"Unique veg types ({mod_data['veg'].nunique()}): {sorted(mod_data['veg'].unique())}")

Domain check OK — no Water/Urban/Agriculture rows.
Unique veg types (14): ['Barren ', 'Native Chapparal ', 'Native Coastal Sage Scrub ', 'Native Conifer Alpine ', 'Native Conifer Forest ', 'Native Desert ', 'Native Grassland ', 'Native Inland Scrub ', 'Native Oak Woodland ', 'Native Wetland ', 'Non-native forest ', 'Non-native grassland ', 'Non-native shrub ', 'Riparian ']


---

## 5. Vegetation Cleaning

Three sequential steps to arrive at the 6 vegetation groups used in the model.

| Step | Action | Reason |
|------|--------|--------|
| Remove Riparian | Drop rows | Atypical fire behaviour; not modelled |
| Map to groups | 13 fine classes → 9 groups | Reduce sparsity |
| Drop Desert/Wetland/Barren | Drop rows | Too few fire events for reliable prediction |
| Final groups | Shrub, Chaparral, Conifer Forest, Conifer Alpine, Grassland, Oak Woodland | 6 groups used in model |

In [6]:
# Step 1: Remove Riparian
n_before = len(mod_data)
mod_data = mod_data[~mod_data['veg'].str.contains('Riparian')]
print(f"Removed Riparian : {n_before - len(mod_data):,} rows")

# Step 2: Map fine veg classes to groups
mod_data['veg'] = mod_data['veg'].str.strip()
mod_data['veg_group'] = mod_data['veg'].map(VEG_MAPPING)

unmapped = mod_data['veg_group'].isnull().sum()
assert unmapped == 0, f"{unmapped} veg types not in VEG_MAPPING — update the mapping"
print(f"Veg groups ({mod_data['veg_group'].nunique()}): {sorted(mod_data['veg_group'].unique())}")

# Step 3: Drop Desert, Wetland, Barren
n_before = len(mod_data)
mod_data = mod_data[~mod_data['veg_group'].isin(REMOVE_VEG_GROUPS)]
print(f"Removed {REMOVE_VEG_GROUPS}: {n_before - len(mod_data):,} rows")
print(f"Final veg groups ({mod_data['veg_group'].nunique()}): {sorted(mod_data['veg_group'].unique())}")

Removed Riparian : 88,236 rows
Veg groups (9): ['Barren', 'Chaparral', 'Conifer Alpine', 'Conifer Forest', 'Desert', 'Grassland', 'Oak Woodland', 'Shrub', 'Wetland']
Removed ['Desert', 'Wetland', 'Barren']: 306,369 rows
Final veg groups (6): ['Chaparral', 'Conifer Alpine', 'Conifer Forest', 'Grassland', 'Oak Woodland', 'Shrub']


---

## 6. Derive `water_year`

Water year convention: October through September.
For example, October 1994 – September 1995 = water year **1995**.

```
water_year = year_month.year + 1   if month >= 10
           = year_month.year       if month < 10
```

In [7]:
mod_data['water_year'] = mod_data['year_month'].apply(
    lambda x: x.year + 1 if x.month >= 10 else x.year
)

summary = mod_data.groupby('water_year').agg(
    month_min  =('year_month', 'min'),
    month_max  =('year_month', 'max'),
    n_rows     =('year_month', 'count'),
    n_fires    =('IS_FIRE',    'sum'),
    fire_rate  =('IS_FIRE',    'mean'),
).reset_index()
print(summary.to_string(index=False))

 water_year month_min month_max  n_rows  n_fires  fire_rate
       1994   1994-01   1994-09  105269     2905   0.027596
       1995   1994-10   1995-09  140566     2644   0.018810
       1996   1995-10   1996-09  140279     3608   0.025720
       1997   1996-10   1997-09  140423     3252   0.023159
       1998   1997-10   1998-09  140779     2249   0.015975
       1999   1998-10   1999-09  140659     3420   0.024314
       2000   1999-10   2000-09  141043     3124   0.022149
       2001   2000-10   2001-09  141489     3056   0.021599
       2002   2001-10   2002-09  141563     3073   0.021708
       2003   2002-10   2003-09  141551     2736   0.019329
       2004   2003-10   2004-09  141562     3114   0.021997
       2005   2004-10   2005-09  141481     2613   0.018469
       2006   2005-10   2006-09  141372     3183   0.022515
       2007   2006-10   2007-09  140698     3927   0.027911
       2008   2007-10   2008-09  140505     3495   0.024875
       2009   2008-10   2009-09  140897 

## 7. Final QA

Check shape, missing rates, and fire rate before saving.

In [8]:
print(f"Final shape : {mod_data.shape}")
print(f"Fire rate   : {mod_data['IS_FIRE'].mean():.4f}")

missing = mod_data.isnull().mean().mul(100)
missing = missing[missing > 0].sort_values(ascending=False)
if missing.empty:
    print("No missing values.")
else:
    print("Missing rate (%) — non-zero columns only:")
    print(missing.to_string())

print(f"\nColumns ({mod_data.shape[1]}):")
print(list(mod_data.columns))

Final shape : (3773561, 68)
Fire rate   : 0.0194
No missing values.

Columns (68):
['year_month', 'lat', 'lon', 'SWE', 'SWE_min', 'SWE_max', 'dead_fuel_moisture_1000hr', 'dead_fuel_moisture_1000hr_min', 'dead_fuel_moisture_1000hr_max', 'dead_fuel_moisture_100hr', 'dead_fuel_moisture_100hr_min', 'dead_fuel_moisture_100hr_max', 'max_air_temperature', 'max_air_temperature_min', 'max_air_temperature_max', 'max_relative_humidity', 'max_relative_humidity_min', 'max_relative_humidity_max', 'min_air_temperature', 'min_air_temperature_min', 'min_air_temperature_max', 'min_relative_humidity', 'min_relative_humidity_min', 'min_relative_humidity_max', 'precipitation_amount', 'precipitation_amount_min', 'precipitation_amount_max', 'specific_humidity', 'specific_humidity_min', 'specific_humidity_max', 'surface_downwelling_shortwave_flux_in_air', 'surface_downwelling_shortwave_flux_in_air_min', 'surface_downwelling_shortwave_flux_in_air_max', 'wind_speed', 'wind_speed_min', 'wind_speed_max', 'populat

## 8. Save Output

In [9]:
out_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE)
mod_data.to_parquet(out_path, index=False)

print(f"Saved -> {out_path}")
print(f"File size  : {os.path.getsize(out_path)/1e9:.2f} GB")
print(f"Final shape: {mod_data.shape[0]:,} rows × {mod_data.shape[1]} cols")

Saved -> E:\zcao\CA_Wildfire\Clean_Data\Model_Data\Evaluation\Features_w_Label\Monthly_03152026\monthly_model_data_by_year.parquet
File size  : 0.33 GB
Final shape: 3,773,561 rows × 68 cols


## 9. Summary

| Step | Description | Key Result |
|------|-------------|------------|
| Load | 4 input tables from `03_01`, `02_06`, `03_03` | Shapes confirmed |
| Merge | weather → wind → fire label → static | Row count stable |
| Domain check | Assert no Water/Urban/Agriculture | Passed |
| Remove Riparian | Drop Riparian veg rows | Atypical fire behaviour excluded |
| Map veg groups | 13 fine classes → 9 groups | `veg_group` column added |
| Drop sparse groups | Remove Desert, Wetland, Barren | 6 groups remain |
| Water year | Oct(Y)–Sep(Y+1) → `water_year` = Y+1 | Column added |
| Save | `Monthly_03152026/monthly_model_data_by_year.parquet` | Input for `03_04` |